In [36]:
# Generamos una estructura de datos de texto plano de ejemplo, se puede abrir un archivo CSV o cualquier texto plano

archivo = """
FACTURA 001 | CLIENTE: Juan Perez | MONTO: 1500 | TIPO: ORIGINAL
FACTURA 002 | CLIENTE: ACME SA | MONTO: 3200 | TIPO: DUPLICADO
FACTURA 003 | CLIENTE: Juan Perez | MONTO: 2100 | TIPO: ORIGINAL
ORDEN 1001 | CLIENTE: ACME SA | ESTADO: PENDIENTE
ORDEN 1002 | CLIENTE: Juan Perez | ESTADO: APROBADA | FACTURA : 001
ORDEN 1003 | CLIENTE: ACME SA | ESTADO: PENDIENTE
ORDEN 1004 | CLIENTE: Juan Perez | ESTADO: APROBADA  | FACTURA : 003
ORDEN 1005 | CLIENTE: Juan Perez | ESTADO: APROBADA
"""

In [1]:
# Instalamos la libreria groq
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.4 MB/s eta 0:00:00


In [37]:
 # Importamos la libreria de colab para poder usar el key_vault , si se ejecuta en local se puede optar por usar un .env
from google.colab import userdata
api_key = userdata.get('GROQ_API_KEY')
 # Importamos la libreria para comunicarnos con la api
from groq import Groq

#Creamos la conexion con la funcion Groq
client = Groq(api_key=api_key)


In [38]:
# Importamos la libreria de Groq
from groq import Groq

# Conexión
client = Groq(api_key=api_key)

# Prompt del sistema (instrucciones)
system_prompt = """
Eres un asistente especializado en extraer información estructurada desde consultas en lenguaje natural.

Tu tarea es analizar la consulta del usuario y extraer los posibles identificadores relacionados con:

- usuario (nombre, cliente o entidad)
- orden (número o identificador de orden)
- factura (número o identificador de factura)

Reglas:
- Devuelve únicamente un JSON válido
- No incluyas explicaciones ni texto adicional
- Si un campo no está presente, devuelve null
- No inventes valores
- Mantén los valores exactamente como aparecen en el texto

Formato de salida:
{
  "usuario": null,
  "orden": null,
  "factura": null
}
"""



In [43]:
# Input del usuario
pregunta = "Juan Perez tiene ordenes APROBADAS sin facturar?"

# Llamada a la API
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": pregunta}
    ]
)



In [44]:
import re
import json

def regex_buscar_inclusivo(archivo, filtros):
    """
    Busca coincidencias en un texto línea por línea usando OR entre:
    usuario, orden, factura

    Parámetros:
    - archivo: string con múltiples líneas (documentos)
    - filtros: dict con keys usuario, orden, factura (pueden ser None)

    Retorna:
    - lista de líneas que coinciden
    """

    usuario = filtros.get("usuario")
    orden = filtros.get("orden")
    factura = filtros.get("factura")

    patrones = []

    # Construimos patrones solo si no son None
    if usuario:
        patrones.append(rf"\b{re.escape(usuario)}\b")

    if orden:
        patrones.append(rf"\b{re.escape(str(orden))}\b")

    if factura:
        patrones.append(rf"\b{re.escape(str(factura))}\b")

    # Si no hay filtros, devolvemos vacío
    if not patrones:
        return []

    # OR entre todos los patrones
    regex = "|".join(patrones)

    resultados = []

    for linea in archivo.split("\n"):
        if re.search(regex, linea):
            resultados.append(linea)

    return resultados

In [45]:
# Convertimos en Json los parametros determinados por el LLM a partir de la consulta del usuario en lenguaje natural

filtros = response.choices[0].message.content
filtros = json.loads(filtros)

# Buscamos los registros que coincidan con los parametros encontrados
contexto =  regex_buscar_inclusivo(archivo, filtros)

In [46]:
# Ejecutamos la nueva llamada al LLM con el contexto ampliado
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": f"""
Usa el siguiente contexto para responder:

{contexto}

PREGUNTA:
{pregunta}
"""
        }
    ]
)

#Imprimimos la respuesta del modelo
print(response.choices[0].message.content)


Para resolver esto, debemos analizar el conjunto de datos proporcionado. Tenemos dos facturas y tres órdenes. Las facturas son:

- FACTURA 001
- FACTURA 003

Y las órdenes son:

- ORDEN 1002 (referenciada a la factura 001 y estado: APROBADA)
- ORDEN 1004 (referenciada a la factura 003 y estado: APROBADA)
- ORDEN 1005 (Estado: APROBADA pero no tiene ninguna factura asociada)

Dado que las órdenes 1002 y 1004 tienen facturas asociadas y ya han sido facturadas, la respuesta es que no, Juan Perez no tiene órdenes APROBADAS sin facturar.

Sin embargo, la ORDEN 1005 SIQUE cuenta como una APROBADA sin que tenga ninguna factura asociada
